<div>
    <h3>Aligning with DPO a phi3-3 model.</h3>    
</div>

<h4>Install Dependencies and requirments</h4

In [1]:
!pip install -q datasets==2.19.1
!pip install -q trl==0.8.6
!pip install -q peft==0.11.1
!pip install -q transformers==4.41.0
!pip install -q bitsandbytes==0.43.1
!pip install -q sentencepiece==0.1.99
!pip install -q accelerate==0.30.1
!pip install -q huggingface_hub==0.23.1

In [2]:

!pip uninstall -y bitsandbytes triton
!pip install bitsandbytes

Found existing installation: bitsandbytes 0.43.1
Uninstalling bitsandbytes-0.43.1:
  Successfully uninstalled bitsandbytes-0.43.1
Found existing installation: triton 3.6.0
Uninstalling triton-3.6.0:
  Successfully uninstalled triton-3.6.0
  Using cached bitsandbytes-0.50.0-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (1.7 kB)
Using cached bitsandbytes-0.50.0-py3-none-manylinux_2_24_x86_64.whl (40.9 MB)
Using cached triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (188.3 MB)


In [ ]:
# import libraries
import torch
import gc
import transformers
from transformers import AutoModelForCausalLM,AutoTokenizer
from transformers import TrainingArguments,BitsAndBytesConfig
from datasets import load_dataset
from peft import LoraConfig,get_peft_model ,PeftModel
from trl import DPOTrainer
import bitsandbytes as bnb
from getpass import getpass

<h4 style="color:yellow">Another necessary step is to log in to Hugging Face.</h4>

In [4]:
hf_token=getpass("Hugging face password...")


Hugging face password...··········


In [5]:
! huggingface-cli login --token $hf_token

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: read).
Your token has been saved to /root/.cache/huggingface/token
Login successful


## Format dataset

The model I've chosen is the Microsoft Phi-3 mini with the 4k context. It's a 3.8B parameter model that is very competitive and in many cases outperforms 7B parameter models.
I've chosen a small model so that its training can be done with few resources on Google Colab or on a not very large GPU.


In [6]:
model_name="microsoft/Phi-3-mini-4k-instruct"
new_model="phi-3-mini-dpo-apress"


In [7]:
#tokenizer
tokenizer=AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token=tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Before you begin training the model, is necesary need to load the dataset and transform it to fit the format required by the DPOTrainer class, that consists of three fields: the prompt, the chosen answer, and a discarded answer.

In [8]:
#Load dataset
dataset_original =  load_dataset("argilla/distilabel-capybara-dpo-7k-binarized",
                                   split='train[300:2500]')
dataset_eval = load_dataset("argilla/distilabel-capybara-dpo-7k-binarized",
                               split='train[:300]')

#To save column_names
orginal_columns=dataset_original.column_names
print(orginal_columns)

['source', 'conversation', 'original_response', 'generation_prompt', 'raw_generation_responses', 'new_generations', 'prompt', 'chosen', 'rejected', 'rating_chosen', 'rating_rejected', 'chosen_model', 'rejected_model']


In [9]:
dataset_original

Dataset({
    features: ['source', 'conversation', 'original_response', 'generation_prompt', 'raw_generation_responses', 'new_generations', 'prompt', 'chosen', 'rejected', 'rating_chosen', 'rating_rejected', 'chosen_model', 'rejected_model'],
    num_rows: 2200
})

In [10]:
dataset_filtered=dataset_original.filter(
    lambda r:r["rating_chosen"]>=4.5 and r["rating_rejected"]<=2.5
)

This first filter only retrieves those rows where the rating of the chosen response is very high and the rating of the discarded responses is very low. This is a way to facilitate the model's learning, although it's also possible that it doesn't help in the last epochs of training.

I'm going to perform a second filter to keep the prompt length under control, as the selected model only accepts a length of 4000 tokens.

In [11]:
dataset_filtered = dataset_filtered.map(lambda r: {"messages": len(r["chosen"])}).filter(lambda r: r["messages"]<3 and len(r["prompt"]) + len(r["chosen"]) + len(r["rejected"]) < 3800)


In [12]:
dataset_filtered

Dataset({
    features: ['source', 'conversation', 'original_response', 'generation_prompt', 'raw_generation_responses', 'new_generations', 'prompt', 'chosen', 'rejected', 'rating_chosen', 'rating_rejected', 'chosen_model', 'rejected_model', 'messages'],
    num_rows: 169
})

In [13]:
dataset_eval_filtered=dataset_eval.filter(
    lambda r:r["rating_chosen"]>=4.5 and r["rating_rejected"]<=2.5
)

In [14]:
def chatml_format(example):
      # get everything except the last message as input
      prompt=tokenizer.apply_chat_template(example["chosen"][:-1],tokenize=False,add_generation_prompt=True)
      # get the last assistant responses
      chosen=example["chosen"][-1]["content"]+"<|end|>\n"
      rejected=example["rejected"][-1]["content"]+"<|end|>\n"


      return {
          "prompt":prompt,
          "chosen":chosen,
          "rejected":rejected
      }



In [15]:
orginal_columns

['source',
 'conversation',
 'original_response',
 'generation_prompt',
 'raw_generation_responses',
 'new_generations',
 'prompt',
 'chosen',
 'rejected',
 'rating_chosen',
 'rating_rejected',
 'chosen_model',
 'rejected_model']

I'll use the dataset's map function to execute the transformation on each row, and also remove the original columns.

In [16]:
# Format dataset
dataset = dataset_filtered.map(
    chatml_format,
    remove_columns=dataset_filtered.column_names
)
# Print sample
dashline="-".join("" for _ in range(30))

print(dataset[12]["prompt"])
print(dashline)
print("----chosen")
print(dataset[12]["chosen"])
print(dashline)
print("----rejected")
print(dataset[12]["rejected"])


<|user|>
Solve 36146684-304553543134. Only respond with math and no words.<|end|>
<|assistant|>

-----------------------------
----chosen
36146684 - 304553543134 = -304517396450<|end|>

-----------------------------
----rejected
(36146684 / 3134) * (36146684 mod 3134) + (30455354 / 17) * (30455354 mod 17) = 11415845286790903
Alternatively, using prime factorization and the Chinese Remainder Theorem:
36146684 = 2^5 * 9573, 30455354 = 2 * 29 * 4171
36146684 mod 9573 = 4332, 30455354 mod 29 = 13, 30455354 mod 4171 = 3965
(36146684 / 9573) * 4332 + (30455354 / 29) * 13 + (30455354 / 4171) * 3965 = 11415845286790903<|end|>



In [17]:
# Format dataset
dataset_eval = dataset_eval_filtered.map(
    chatml_format,
    remove_columns=orginal_columns
)

#Print sample

print(dataset_eval[12]["prompt"])
print(dashline)
print("----chosen")
print(dataset_eval[12]["chosen"])
print(dashline)
print("----rejected")
print(dataset_eval[12]["rejected"])

<|user|>
Solve: 4(3x - 5) = 8(x + 1)<|end|>
<|assistant|>
Let's solve this step by step:

Given the equation 4(3x - 5) = 8(x + 1), first distribute the 4 and 8 on both sides of the equation:

12x - 20 = 8x + 8

Next, subtract 8x from both sides to isolate x terms on the left side:

4x - 20 = 8

Now, add 20 to both sides to isolate the x term:

4x = 28

Finally, divide both sides by 4 to solve for x:

x = 7

So, the solution to the equation 4(3x - 5) = 8(x + 1) is x = 7.<|end|>
<|user|>
Given the solution x = 7, how would you utilize this value to find the y-intercept of the linear equation y = 3x - 2?<|end|>
<|assistant|>
The y-intercept of a linear equation is the point where the line crosses the y-axis. This is the value of y when x equals 0. 

In the equation y = 3x - 2, the y-intercept is already given as -2. The value of x = 7 doesn't affect the y-intercept because the y-intercept is determined solely by the constant term in the equation, which is -2 in this case. 

So, the y-inte

## Train model with DPO

## Finetuning with DPOTrainer.

Time to start working with the necessary configurations to perform alignment using DPO.



In [18]:
#lora configuration
peft_config=LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.01,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear"

)

The value of **r** indicates the size of the reparameterization; the higher the value, the more parameters are trained. An 8 is at the upper limit of what is recommended for small models.

To further accentuate the weight of the new training, I use the **lora_alpha** value. It's a multiplier that adjusts the layers inserted by LoRA. Normally it's left at 1, but in the case of DPO, I've seen values as high as 128.

The recommendation is that **lora_alpha** should be double the value of **r**. Since **r** varies depending on the model size, you may end up with a very high lora_alpha value if you want to fine-tune a large model and, for example, specify an **r** of 64.

In [19]:
bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

The quantization configuration holds no secrets, we are reducing the model's precision to 4 bits.

In [20]:
#Model to Fine-tune
model =AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
)
model.config.use_cache=False

`low_cpu_mem_usage` was None, now set to True since model is quantized.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [20]:
#Flush memory
#del dpo_trainer, model
gc.collect()
torch.cuda.empty_cache()

In [21]:
training_args = TrainingArguments(
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    learning_rate=5.0e-06,
    eval_strategy="epoch",
    logging_strategy="epoch",
    lr_scheduler_type="cosine",
    num_train_epochs=6,
    save_strategy="epoch",
    logging_steps=1,
    output_dir=new_model,
    optim="paged_adamw_32bit",
    warmup_steps=2,
    bf16=True,
    report_to="none",
)


In [ ]:
Trainer=DPOTrainer(
    model=model,
    args=TrainingArguments,
    train_dataset=dataset,
    eval_dataset=dataset_eval,
    tokenizer=tokenizer


)
Trainer.train()

/usr/local/lib/python3.12/dist-packages/trl/trainer/dpo_trainer.py:300: UserWarning: `max_length` is not set in the DPOTrainer's init it will default to `512` by default, but you should do it yourself in the future.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/dpo_trainer.py:307: UserWarning: `max_prompt_length` is not set in the DPOTrainer's init it will default to `128` by default, but you should do it yourself in the future.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/dpo_trainer.py:332: UserWarning: When using DPODataCollatorWithPadding, you should set `remove_unused_columns=False` in your TrainingArguments we have set it for you, but you should do it yourself in the future.
  warnings.warn(


## Upload model

In [1]:
model_path="final_checkpoint_Phi-3_dpo"

In [ ]:
#save model and tokenizer 
Trainer.model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

In [ ]:
# to produce using memory cache we must empty cache 
gc.collect()
torch.cuda.empty_cache()

Now, you're going to load the original model again, but this time in its unquantized format.

In [ ]:
base_model=AutoModelForCausalLM.from_pretrained(
    model_name,
    return_dict=True,
    torch_dtype=torch.bfloat16 
)
tokenizer=AutoTokenizer.from_pretrained(model_name)


The original model and the saved training are being merged.

In [ ]:
model=PeftModel.from_pretrained(base_model,model_path)
model=model.merge_and_unload()


In [ ]:
model.save_pretrained(new_model)
tokenizer.save_pretrained(new_model)


## Inference

Let's test the new model and compare with the original

In [3]:
# Format Prompt
message=[
    {"role": "user", "content": "3713841893836/4?\nLimit your response to mathematical expressions and symbols."}

]

In [ ]:
tokenizer_new_model=AutoTokenizer.from_pretrained(new_model)
prompt=tokenizer.new_model.apply_chat_template(message, add_generation_prompt=True, tokenize=False)

#create pipline
pipeline_new=transformers.pipeline(
    "text-generation",
    model=new_model,
    tokenizer=tokenizer_new_model
)

In [ ]:
Sequences=pipeline_new(
    prompt,
    do_sample=True,
    temperature=0.1,
    top_p=0.2,
    num_return_sequences=1,
    max_length=200,

)
print(Sequences[0]["generated_text"])
